# Performance Measures for Message Digests, Symmetric and Asymmetric Cryptography

#### PL2G10

Adam Krivka - 202512248

Gonçalo Reimão Silva da Luz - 202205522

Rafael Arruda Costa - 202300278

## Experimental Setup and Benchmarking Methodology

For measurements of the execution time, we have used the Python `timeit` module.

- **Setup:**  
  The benchmarks were run on a **Windows** machine with a **R7 5800x3D** processor and **32 GB** of RAM, running Python **3.13.5**.
- **Isolation:**  
  To isolate just the crypto operations, all files were read into memory before starting the timer. The tested file sizes were 8, 64, 512, 4096, 32768, 262144, and 2097152 bytes.

- **Statistical Significance:**  
  Each operation (encryption and decryption) was repeated **100 times** for every file size. A file of each size is then encrypted for decryption benchmarking and the measurement is repeated a hundred times as well. We then computed the mean and standard deviation across runs to make sure the results were consistent. The times were measured in seconds and converted to microseconds for plotting.


# File Generation (Part A)

In [ ]:
import os

sizes = [8,64,512,4096,32768,262144,2097152]

for s in sizes:
    with open(f"file_{s}.bin","wb") as f:
        f.write(os.urandom(s))

This code was used to generate binary files of the requested byte sizes. Binary files were used so the input consists of uniformly random byte sequences, avoiding any encoding-related bias that could arise from text-based files.

# AES Counter Mode Hybrid Encryption and Decryption (Part B)

In [ ]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

def encrypt_ctr(key, plaintext):
    nonce = os.urandom(16)
    encryptor = Cipher(
        algorithms.AES(key),
        modes.CTR(nonce),
    ).encryptor()
    ciphertext = encryptor.update(plaintext) + encryptor.finalize()
    return (nonce, ciphertext)

def decrypt_ctr(key, nonce, ciphertext):
    decryptor = Cipher(
        algorithms.AES(key),
        modes.CTR(nonce),
    ).decryptor()
    return decryptor.update(ciphertext) + decryptor.finalize()

This code was developed by modifying the AES-GCM (Galois/Counter Mode) example from the `cryptography.hazmat.primitives` library. Since the Galois method includes an additional step to authenticate data and detect alterations, we removed the authentication tags. Lastly, we set the nonces for both encryption and decryption to 16 bytes, which matches the standard block size required for the AES state transformation matrix.

##### Benchmarking

To generate the following data, we utilized a helper file (utils.py) to calculate the mean, standard deviation and confidence intervals. We applied a 95% confidence level using the z-critical value of 1.96, given our sample size of 100 iterations per test.

In [ ]:
def calculate_statistics(data_list, confidence_level=0.95)
    
    return {
    "mean": mean,
    "stdev": stdev,
    "ci_low": mean - margin_of_error,
    "ci_high": mean + margin_of_error
}

The performance test function iterates through each file size 100 times to mitigate system noise. The resulting encryption and decryption times are stored in lists, which are then used to calculate the mean, standard deviation and 95% confidence intervals for the graph generation.

##### Fixed files first run

|File Name            | Mean (us)  | StdDev     | 95% CI                     |
|---------------------|------------|------------|----------------------------|
|file_8.bin           | 22.7480    | 106.3888   | [1.8958, 43.6002] (Enc)    |
|                     | 10.2770    | 3.5021     | [9.5906, 10.9634] (Dec)    |
|file_64.bin          | 12.3030    | 11.9809    | [9.9547, 14.6513] (Enc)    |
|                     | 10.2310    | 5.0966     | [9.2321, 11.2299] (Dec)    |
|file_512.bin         | 11.0460    | 3.2811     | [10.4029, 11.6891] (Enc)   |
|                     | 9.7260     | 1.5418     | [9.4238, 10.0282] (Dec)    |
|file_4096.bin        | 11.5200    | 2.1685     | [11.0950, 11.9450] (Enc)   |
|                     | 10.6450    | 1.5218     | [10.3467, 10.9433] (Dec)   |
|file_32768.bin       | 16.0920    | 3.9567     | [15.3165, 16.8675] (Enc)   |
|                     | 14.6010    | 2.4968     | [14.1116, 15.0904] (Dec)   |
|file_262144.bin      | 66.7170    | 11.7075    | [64.4223, 69.0117] (Enc)   |
|                     | 48.7950    | 6.3705     | [47.5464, 50.0436] (Dec)   |
|file_2097152.bin     | 841.7220   | 34.7284    | [834.9152, 848.5288] (Enc) |
|                     | 829.5930   | 28.0552    | [824.0942, 835.0918] (Dec) |

<img src="aes_combined_plot_fixed.png" width="60%" alt="AES Performance Graph">

##### Fixed files second run

|File Name            | Mean (us)  | StdDev     | 95% CI                     |
|---------------------|------------|------------|----------------------------|
|file_8.bin           | 21.7010    | 106.4498   | [0.8368, 42.5652] (Enc)    |
|                     | 9.7610     | 2.3105     | [9.3081, 10.2139] (Dec)    |
|file_64.bin          | 12.6090    | 4.0707     | [11.8111, 13.4069] (Enc)   |
|                     | 9.7520     | 1.5434     | [9.4495, 10.0545] (Dec)    |
|file_512.bin         | 11.2760    | 3.1839     | [10.6519, 11.9000] (Enc)   |
|                     | 10.4740    | 3.8080     | [9.7276, 11.2204] (Dec)    |
|file_4096.bin        | 11.6400    | 2.3151     | [11.1862, 12.0938] (Enc)   |
|                     | 11.3280    | 3.3258     | [10.6761, 11.9799] (Dec)   |
|file_32768.bin       | 17.4110    | 6.2687     | [16.1823, 18.6397] (Enc)   |
|                     | 15.3060    | 2.6056     | [14.7953, 15.8167] (Dec)   |
|file_262144.bin      | 70.3540    | 19.4875    | [66.5344, 74.1736] (Enc)   |
|                     | 49.5680    | 7.9695     | [48.0060, 51.1300] (Dec)   |
|file_2097152.bin     | 912.7800   | 231.8198   | [867.3433, 958.2167] (Enc) |
|                     | 869.3980   | 141.6342   | [841.6377, 897.1583] (Dec) |

<img src="aes_combined_plot_fixed_2.png" width="60%" alt="AES Performance Graph">

When comparing the results, we can see that even when using the same set of fixed files, the execution times vary. While the overall performance trend maintains its shape and scale, the standard deviation quantifies these variances. Although the results are not identical, they are not statistically different because their confidence intervals overlap (WIREs Comp Stat 2012, 4:98–106. doi: 10.1002/wics.192), except for the largest file. This is most prominent in the smallest and largest file sizes due to system noise.

##### Random files first run

|File Name            | Mean (us)  | StdDev     | 95% CI                     |
|---------------------|------------|------------|----------------------------|
|file_8.bin           | 20.8570    | 104.1228   | [0.4489, 41.2651] (Enc)    |
|                     | 9.6750     | 2.2401     | [9.2359, 10.1141] (Dec)    |
|file_64.bin          | 10.6470    | 3.6168     | [9.9381, 11.3559] (Enc)    |
|                     | 9.6130     | 2.2640     | [9.1693, 10.0567] (Dec)    |
|file_512.bin         | 10.5840    | 2.1986     | [10.1531, 11.0149] (Enc)   |
|                     | 9.7500     | 1.4038     | [9.4749, 10.0251] (Dec)    |
|file_4096.bin        | 18.8440    | 2.5487     | [18.3445, 19.3435] (Enc)   |
|                     | 20.0170    | 5.0918     | [19.0190, 21.0150] (Dec)   |
|file_32768.bin       | 17.4540    | 5.8434     | [16.3087, 18.5993] (Enc)   |
|                     | 15.5220    | 4.3276     | [14.6738, 16.3702] (Dec)   |
|file_262144.bin      | 76.5420    | 28.8014    | [70.8969, 82.1871] (Enc)   |
|                     | 50.0840    | 7.0217     | [48.7077, 51.4603] (Dec)   |
|file_2097152.bin     | 817.8750   | 27.2715    | [812.5298, 823.2202] (Enc) |
|                     | 856.4070   | 71.6526    | [842.3631, 870.4509] (Dec) |

<img src="aes_combined_plot_random.png" width="60%" alt="AES Performance Graph">

##### Random files second run

|File Name            | Mean (us)  | StdDev     | 95% CI                     |
|---------------------|------------|------------|----------------------------|
|file_8.bin           | 21.1160    | 106.1096   | [0.3185, 41.9135] (Enc)    |
|                     | 9.7570     | 2.2688     | [9.3123, 10.2017] (Dec)    |
|file_64.bin          | 11.5660    | 3.2320     | [10.9325, 12.1995] (Enc)   |
|                     | 11.9640    | 3.4264     | [11.2924, 12.6356] (Dec)   |
|file_512.bin         | 11.0910    | 3.7601     | [10.3540, 11.8280] (Enc)   |
|                     | 9.7340     | 1.6593     | [9.4088, 10.0592] (Dec)    |
|file_4096.bin        | 12.8260    | 4.3648     | [11.9705, 13.6815] (Enc)   |
|                     | 10.6690    | 1.7322     | [10.3295, 11.0085] (Dec)   |
|file_32768.bin       | 18.3150    | 6.2228     | [17.0953, 19.5347] (Enc)   |
|                     | 14.9360    | 3.0083     | [14.3464, 15.5256] (Dec)   |
|file_262144.bin      | 75.7280    | 16.1734    | [72.5580, 78.8980] (Enc)   |
|                     | 56.2100    | 10.7207    | [54.1087, 58.3113] (Dec)   |
|file_2097152.bin     | 884.6110   | 124.9443   | [860.1219, 909.1001] (Enc) |
|                     | 827.7150   | 25.1642    | [822.7828, 832.6472] (Dec) |

<img src="aes_combined_plot_random_2.png" width="60%" alt="AES Performance Graph">

In the analysis of randomly generated files, we observe a similar pattern. For most sizes, the confidence intervals overlap, indicating that the results are statistically indistinguishable. However, significant divergences appear in larger files, specifically at the 4096 and 2097152 bytes files. This observation suggests that system noise and resource contention increase during longer execution periods, regardless of the data's randomness.

** The complete Python implementation is included in the attached file.*


# RSA-Based Hybrid Encryption and Decryption (Part C)

## Pre-Implementation and verification

We assumed the correctness of the textbook RSA property, where $(m^e)^d \pmod n = m$. As long as we got the correctly formatted the random integer r, it is guaranteed it will be safely recovered.

We mapped out everything the receiver would need to rebuild the original message as the sender encrypted it. The receiver uses their private RSA key for decrypting the secret seed ($r$) safely. Because hash functions always give the same output for the same input, the receiver can feed the recovered seed ($r$) and the block number ($i$) into SHA-256, which guarantees they recreate the exact same keystream ($H(i, r)$), which the sender used.

As both the sender and receiver can build the exact same keystream, the decryption should logically follow. By the nature of XOR operation, combining the ciphertext with keystream a second time reverses the encryption and gives back the original file block.

We also identified that files rarely have the size that are exact multiples of 32 B, so we made sure the math would work for those cases as well.

## Code Implementation and Explanation

In this part of the assignment, we implemented a hybrid encryption scheme using RSA for handling key encryption and SHA-256 for securing data itself.

---

### **Key Generation and Core RSA Functions**

We used the `cryptography.hazmat.primitives` library to generate a 2048-bit RSA key pair, which is considered to be secure. The public exponent was set to 65537, which is a common choice for its computational efficiency in binary.

From the generated key we extracted the components ($n$, $e$, and $d$) and implemented RSA manually using Python’s built-in `pow()` function for modular exponentiation:

- **RSA Encrypt:**  
  $c = m^e \pmod n$

- **RSA Decrypt:**  
  $m = c^d \pmod n$

In [ ]:
# Key generation
generated_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = generated_key.public_key()

d = generated_key.private_numbers().d 
n = public_key.public_numbers().n 
e = public_key.public_numbers().e 

def rsa_encrypt(m_int, e, n):
    return pow(m_int, e, n)

def rsa_decrypt(c_int, d, n):
    return pow(c_int, d, n)



### **Hybrid Encryption Algorithm ($Enc(m; r)$)**

 RSA on itself is too slow for encrypting large files, so we used a hybrid approach: encrypt a small random key with RSA, and use that key to encrypt the actual data.

$$
Enc(m; r) = (RSA(r), H(0, r) \oplus m_0, \dots, H(n, r) \oplus m_n)
$$

The implementation works as follows:

1. **Key Encapsulation:**  
   A random 32-byte (256-bit) value $r$ is generated using `os.urandom(32)` as a one-time session key. We convert it to an integer, encrypt it using RSA, and store it as a 256-byte value (to match the 2048-bit key size).

In [ ]:
# Generate 256-bit session key and encapsulate via RSA
r_bytes = os.urandom(32)
r_int = int.from_bytes(r_bytes, 'big')
encrypted_r_bytes = rsa_encrypt(r_int, e, n).to_bytes(256, 'big')

2. **Data Encapsulation (Stream Cipher):**  
   SHA-256 is used to simulate a stream cipher. The data gets split into 32-byte blocks, and for each block $m_i$, we concatenate the block index $i$ with the key $r$, hash them together, and use the result as a keystream. This keystream is then XORed ($\oplus$) with the data block.


In [ ]:
# For each 32 B plaintext-block m_i:
hash_input = block_num_bytes + r_bytes 
keystream_block = hashlib.sha256(hash_input).digest()

# Truncate it to block length and XOR with plaintext
keystream = keystream_block[:len(m_i)]
c_i = bytes(k ^ m for k, m in zip(keystream, m_i)) 
ciphertext_blocks.append(c_i)


3. **Decryption:**  
   First, we take the first 256 bytes of the ciphertext and decrypt it using the private key $d$ to get back $r$. From there, we generate the same SHA-256 keystream and XOR it with the remaining ciphertext blocks to recover the original file.

** The full Python implementation is included in the attached file.*

---

## Performance Results

The plots below show encryption and decryption times for different file sizes.

### Encryption and Decryption Plot

<img src="rsa_public_combined_plot.png" width="60%" alt="RSA Encryption Plot">

*Figure: RSA-based encryption times (X-axis: bytes, Y-axis: μs)*

---

*NOTE: Docstring briefs for this part were generated by Claude from the source code and graph plotting function was partly generated by Gemini, mainly the design part.*

# SHA Encryption and Decryption (Part D)

In [ ]:
import hashlib
import timeit
# import statistics

def sha256_file(filepath, runs=100):
    with open(filepath, "rb") as f:
        data = f.read()

    timer = timeit.Timer(lambda: hashlib.sha256(data).digest())

    # Warm-up
    timer.timeit(number=50)

    raw_times = timer.repeat(repeat=runs, number=10)

    return [(t / 10) * 1000000 for t in raw_times]

The function reads the entire file into memory and measures the execution time of the SHA-256 hashing operation on its data. We perform a preliminary warm-up execution using the timing framework to reduce the influence of interpreter initialization effects, CPU caching, and other runtime overheads.

The `timeit.Timer` object is then used to execute the hashing operation multiple times, each being 10 repeated hashes per measurement. The function returns a list of execution times corresponding to a single SHA-256 computation, in microseconds.

This approach isolates the computational cost of the cryptographic hash function from file input/output overhead.

In [ ]:
import sha_crypto
import statistics
import matplotlib.pyplot as plt
import pandas as pd
import math

results = []

print("File Size (B) | Throughput (B/µs) | CI ")
print("--------------+-------------------+--------")

files = [
    (8, "file_8.bin"),
    (64, "file_64.bin"),
    (512, "file_512.bin"),
    (4096, "file_4096.bin"),
    (32768, "file_32768.bin"),
    (262144, "file_262144.bin"),
    (2097152, "file_2097152.bin")
]

for size, f in files:

    runs = 100 if size > 512 else 500

    all_times = sha_crypto.sha256_file(f, runs=runs)

    avg_time = statistics.mean(all_times)
    std_dev = statistics.stdev(all_times)

    throughput = size / avg_time

    n = len(all_times)
    ci = 1.96 * (std_dev / math.sqrt(n))

    ci_rel = ci / avg_time

    print(f"{size:<13} | {throughput:<17.6f} | ±{ci_rel:.4f}")

    results.append({
        "Size": size,
        "MeanTime": avg_time,
        "CI": ci,
    })

This section evaluates SHA-256 performance across different file sizes by repeatedly hashing each file using `sha_crypto.sha256_file()` and collecting execution time samples for each input size.

For file sizes above 512 bytes, 100 independent runs are performed, while for smaller file sizes (8, 64, and 512 bytes), 500 runs are used. This is because the execution times are so short that they fall close to the resolution limit of the timing method. At this scale, fixed overheads like function calls, interpreter execution, and CPU scheduling have a noticeable impact on the results, which leads to higher variance relative to the mean, which means that more samples are needed to get a stable average.

As such, these measurements aren't really representative of SHA-256's asymptotic performance, but they're still useful for seeing where execution shifts from being overhead-dominated (small file sizes) to compute-dominated (larger file sizes). Because of this, results for small file sizes should be taken as rough indicators rather than direct comparisons to the larger input results.

For each file size, the following metrics are calculated:

- **Mean execution time:** average cost of computing the SHA-256 hash for a given file size, used as the primary performance indicator.
- **Standard deviation:** variability across repeated measurements caused by system-level noise such as CPU scheduling and caching effects.
- ***95%* confidence interval:** estimate of the uncertainty in the mean execution time, allowing comparison of whether differences between file sizes are statistically meaningful.
- **Throughput:** represents the processing rate in bytes per microsecond, derived from the ratio of input size to mean execution time, and is used to evaluate scalability.

These metrics allow performance to be interpreted both in terms of absolute execution time and normalized throughput, enabling analysis of how SHA-256 scales with increasing input size.

Finally, the results are stored and passed to a visualization stage using matplotlib and pandas to generate plots of execution time versus input size. The plotting code used to generate the visualizations was assisted by an **AI tool (Google Gemini)**. The implementation was reviewed by us to ensure correctness and alignment with the experimental requirements.

### Example Output Table

| File Size (B) | Throughput (B/µs) | 95% CI (µs) |
|---------------|-------------------|-------------|
| 8             | 11.626893         | ±0.0218     |
| 64            | 99.635706         | ±0.0135     |
| 512           | 426.951287        | ±0.0202     |
| 4096          | 1538.866132       | ±0.0241     |
| 32768         | 2076.315752       | ±0.0102     |
| 262144        | 2178.240038       | ±0.0031     |
| 2097152       | 2197.488674       | ±0.0021     |

This table shows SHA-256 throughput (bytes processed per microsecond) and how uncertain each measurement is. For very small inputs, throughput is much lower because fixed overheads (like function calls and interpreter execution) make up a large portion of the total execution time, as discussed above.

As file size increases, those overheads become less significant and throughput rises quickly. Past around 32 KB it levels off, which is where the actual SHA-256 computation starts to dominate rather than the surrounding overhead.

The confidence intervals also get smaller as file size increases, which makes sense, as larger files take longer to process, so the measurements are less affected by background system noise.

### Plot

<img src="sha256_benchmark.png" width="60%" alt="SHA Digests Generation">

*Figure: SHA digests generation times (X-axis: bytes, Y-axis: $\mu$s)*

The linear plot shows that execution time scales roughly proportionally with file size for larger inputs, which is what we expect from a hash function processing data in fixed-size blocks. Meanwhile, in the log-log plot, the curve isn't a straight line, which tells us the relationship isn't cleanly linear across all scales. The steep rise at small file sizes followed by a gradually flattening slope reflects the shift from overhead-dominated to compute-dominated execution that the throughput table also shows. The error bars are only really visible on the mid-range file sizes, which lines up with what the CI column in the table already suggested.

## Results Analysis and Comparisons (Part E)

### 1. RSA-Based Encryption vs. Decryption Times

**Observation:**  
For all file sizes, RSA-based decryption is much slower than encryption. Both increase linearly as file size grows, but decryption starts with a much higher initial cost. Left part of each graph clearly shows the fixed overhead of the RSA math: It clearly shows the algorithm contains two different phases.

**Explanation:**  
This difference comes from how RSA works mathematically during the key encapsulation step (encrypting/decrypting the session key $r$). Because the stream cipher phase is almost identical for both encryption and decryption, the two lines scale at the same rate, only separated by a constant gap of the initial decryption step.

- **Encryption ($c = r^e \pmod n$):**  
  The public exponent $e = 65537$ is small and has a simple binary form. This makes the computation very quick using the square-and-multiply algorithm.

- **Decryption ($r = c^d \pmod n$):**  
  The private exponent $d$ is a very large number (close to 2048 bits). Performing modular exponentiation with such a large exponent is much more computationally expensive, which slows down decryption significantly. With smaller file sizes, the orange line (Decryption) sits way above the blue line (Encryption), which visually proves that decrypting with a long private key takes significantly more time than encrypting it.

  The linear part of the left graph proves that the SHA-256 hash loop scales with ($O(N)$) complexity, depending on the file size. At larger scales, the RSA math at the beginning of computation is just a tiny bump and hashing and xoring data takes majority of the time.

---

### 2. AES-Based Encryption vs. RSA-Based Encryption

**Observation:**  
When comparing AES-256 (CTR mode) with our hybrid scheme (RSA-2048 + SHA-256), AES is significantly faster, especially for larger files.

**Explanation:**  
Initial Overhead: The hybrid RSA method starts slower because it first has to do a complex 2048-bit calculation to encrypt the session key $r$ securely. Only after that it can start processing the actual file. In contrast, AES just sets up its encryption and starts computing almost immediately.


- **Data Encapsulation:**  
 Both methods take linearly more time as the file gets bigger, but AES is much faster in practice. It’s highly optimized and often supported directly by the CPU. Our RSA hybrid approach works like a stream cipher using SHA-256, but it has to hash every block of data ($H(i, r) \oplus m_i$), which makes it more computationally expensive per byte compared to AES.

---

### 3. AES-Based Encryption vs. SHA Digest Generation

**Observation:**
SHA digest generation is significantly faster for small files, but AES (CTR mode) achieves higher peak throughput as file sizes increase toward 2MB.

**Explanation:**
Initial Overhead: SHA has a much lower "setup" cost for small workloads. In contrast, AES (CTR) requires initializing cipher objects, managing nonces, and setting up counter blocks before encryption begins, which inflates the execution time for small files


- **Peak Efficiency and Hardware Optimization:**
As the file size increases, the fixed overhead becomes negligible compared to the data processing rate. AES (CTR) eventually overtakes SHA because it leverages dedicated hardware instructions (AES-NI) found in modern CPUs. While SHA is highly efficient, the optimized AES-NI pipeline allows for superior B/µs (throughput) when processing large contiguous blocks of data.

- **Statistical Stability:**
The SHA results show extremely narrow confidence intervals, suggesting it is less sensitive to system noise and background interrupts than the AES implementation in this environment. The non-overlapping confidence intervals at the 2MB scale confirm that the performance lead of AES is statistically significant.